# Convolutional Neural Networks: Application (PyTorch 版本)

欢迎来到第四课的第二个作业！本 notebook 保留原作业的主题顺序和教学目标，但将 TensorFlow/Keras 代码替换为 PyTorch。你将完成：

- 使用 `torch.nn.Sequential` 创建笑脸二分类模型；
- 使用自定义 `nn.Module` 的函数式 `forward` 构建手语数字多分类模型；
- 理解卷积层、池化层、Batch Normalization、Flatten 和全连接层；
- 使用 PyTorch 的 `DataLoader`、损失函数、优化器和训练循环完成训练。

**完成本作业后，你应能够：**

- 使用 PyTorch 构建并训练二分类 CNN；
- 使用 PyTorch 构建并训练多分类 CNN；
- 理解 Sequential 模型与自定义 `nn.Module` 的适用场景；
- 理解 PyTorch 中 logits、`Sigmoid`、`CrossEntropyLoss` 之间的关系。

本版本适合学习和运行，不再针对 Coursera 的 TensorFlow AutoGrader。

## 目录

- [1 - 导入包](#1)
    - [1.1 - 加载数据并划分训练集/测试集](#1-1)
- [2 - PyTorch 中的层](#2)
- [3 - Sequential API](#3)
    - [3.1 - 创建 Sequential 模型](#3-1)
        - [练习 1 - happyModel](#ex-1)
    - [3.2 - 训练和评估模型](#3-2)
- [4 - 自定义 Module 与函数式 forward](#4)
    - [4.1 - 加载 SIGNS 数据集](#4-1)
    - [4.2 - 划分训练集/测试集](#4-2)
    - [4.3 - 前向传播](#4-3)
        - [练习 2 - convolutional_model](#ex-2)
    - [4.4 - 训练模型](#4-4)
- [5 - History 字典](#5)
- [6 - 参考资料](#6)

<a name='1'></a>
## 1 - 导入包

首先导入数据处理、绘图和 PyTorch 所需的包。

In [1]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from cnn_utils import *

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def images_to_tensor(x):
    """将课程数据的 NHWC 格式转换为 PyTorch 使用的 NCHW 格式。"""
    return torch.tensor(x, dtype=torch.float32).permute(0, 3, 1, 2)


def binary_labels_to_tensor(y):
    return torch.tensor(np.asarray(y).reshape(-1), dtype=torch.float32)


def class_labels_to_tensor(y):
    return torch.tensor(np.asarray(y).reshape(-1), dtype=torch.long)


def model_summary(model, input_shape):
    """一个轻量的 PyTorch 模型检查器，对应 Keras 的 model.summary()。"""
    print(model)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    with torch.no_grad():
        dummy = torch.zeros(1, *input_shape, device=device)
        output = model(dummy)
    print(f"input shape : {(1, *input_shape)}")
    print(f"output shape: {tuple(output.shape)}")
    print(f"parameters  : {total:,} (trainable: {trainable:,})")


def evaluate_binary(model, loader, criterion):
    model.eval()
    loss_sum = 0.0
    correct = 0
    count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb).reshape(-1)
            loss_sum += criterion(pred, yb).item() * xb.size(0)
            correct += ((pred >= 0.5) == (yb >= 0.5)).sum().item()
            count += xb.size(0)
    return loss_sum / count, correct / count


def train_binary(model, train_loader, test_loader, epochs=10, lr=1e-3):
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb).reshape(-1), yb)
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate_binary(model, train_loader, criterion)
        val_loss, val_acc = evaluate_binary(model, test_loader, criterion)
        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)
        print(f"epoch {epoch + 1:03d}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")
    return history


def evaluate_multiclass(model, loader, criterion):
    model.eval()
    loss_sum = 0.0
    correct = 0
    count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss_sum += criterion(logits, yb).item() * xb.size(0)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            count += xb.size(0)
    return loss_sum / count, correct / count


def train_multiclass(model, train_loader, test_loader, epochs=100, lr=1e-3):
    # CrossEntropyLoss 内部已经包含 LogSoftmax，因此模型输出 logits 即可。
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        train_loss, train_acc = evaluate_multiclass(model, train_loader, criterion)
        val_loss, val_acc = evaluate_multiclass(model, test_loader, criterion)
        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)
        if epoch == 0 or (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            print(f"epoch {epoch + 1:03d}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")
    return history


device: cuda


<a name='1-1'></a>
### 1.1 - 加载数据并划分训练集/测试集

这一部分使用 Happy House 数据集，其中包含人脸图片。任务是判断图片中的人是否在微笑。图片是 **64x64** 的 RGB 图像，因此原始数据是 NHWC 格式，而 PyTorch 卷积层要求 NCHW 格式。

In [2]:
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_happy_dataset()

X_train = images_to_tensor(X_train_orig / 255.)
X_test = images_to_tensor(X_test_orig / 255.)
Y_train = binary_labels_to_tensor(Y_train_orig)
Y_test = binary_labels_to_tensor(Y_test_orig)

print("number of training examples =", X_train.shape[0])
print("number of test examples =", X_test.shape[0])
print("X_train shape =", tuple(X_train.shape))
print("Y_train shape =", tuple(Y_train.shape))
print("X_test shape =", tuple(X_test.shape))
print("Y_test shape =", tuple(Y_test.shape))


NameError: name 'load_happy_dataset' is not defined

你可以显示数据集中的图片。绘图时使用原始的 NHWC 图片，送入 PyTorch 模型时则需要转换为 NCHW。

In [ ]:
index = 124
plt.imshow(X_train_orig[index])
plt.axis("off")
plt.show()


<a name='2'></a>
## 2 - PyTorch 中的层

PyTorch 的每一层通常是 `torch.nn.Module` 的一个对象。层接收输入张量并返回输出张量，多个层可以组合成一个更大的模型。常用的 CNN 层包括 `nn.Conv2d`、`nn.BatchNorm2d`、`nn.ReLU`、`nn.MaxPool2d`、`nn.Flatten` 和 `nn.Linear`。

<a name='3'></a>
## 3 - Sequential API

`nn.Sequential` 适合数据沿着一条直线依次流过各层的模型。它可以把多个层按顺序组合起来，代码简洁，适合本节的笑脸二分类模型。

如果网络存在分支、跳跃连接、多输入或多输出，就应该定义自定义的 `nn.Module`，并在 `forward()` 中显式描述数据流。

<a name='3-1'></a>
### 3.1 - 创建 Sequential 模型

<a name='ex-1'></a>
### 练习 1 - happyModel

实现 `happyModel`，构建下面的模型：

```text
ZEROPAD2D → CONV2D → BATCHNORM → RELU → MAXPOOL → FLATTEN → DENSE
```

参数设置：

- `ZeroPad2d`：四周填充 3，输入大小为 `64 x 64 x 3`；
- `Conv2d`：32 个 `7 x 7` 卷积核，stride 为 1；
- `BatchNorm2d`：对 32 个输出通道进行归一化；
- `ReLU`；
- `MaxPool2d`：使用默认的 `2 x 2` 窗口和 stride 2；
- `Flatten`；
- `Linear`：输出 1 个值，再经过 `Sigmoid` 得到二分类概率。

In [ ]:
# Exercise 1: happyModel

def happyModel():
    """
    ZEROPAD2D -> CONV2D -> BATCHNORM -> RELU -> MAXPOOL -> FLATTEN -> DENSE
    """
    model = nn.Sequential(
        nn.ZeroPad2d(3),
        nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, stride=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2),
        nn.Flatten(),
        nn.Linear(32 * 32 * 32, 1),
        nn.Sigmoid(),
    )
    return model


In [ ]:
happy_model = happyModel().to(device)
model_summary(happy_model, (3, 64, 64))


在 PyTorch 中，通常不调用 Keras 的 `compile()`。我们分别创建损失函数和优化器，并在后面的训练循环中完成前向传播、反向传播和参数更新。笑脸模型最后输出 Sigmoid 概率，因此使用 `BCELoss`。

In [ ]:
happy_train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=16, shuffle=True)
happy_test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size=16, shuffle=False)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(happy_model.parameters(), lr=1e-3)
print(criterion)
print(optimizer)


可以通过打印模型结构、输入输出形状和参数数量，检查模型是否符合预期。`model_summary` 是本 notebook 提供的轻量检查函数。

In [ ]:
model_summary(happy_model, (3, 64, 64))


<a name='3-2'></a>
### 3.2 - 训练和评估模型

PyTorch 不会自动替你执行 `.fit()`。训练循环通常包含以下步骤：

1. 从 `DataLoader` 读取一个 minibatch；
2. 前向传播得到预测值；
3. 计算 loss；
4. 调用 `loss.backward()` 计算梯度；
5. 调用 `optimizer.step()` 更新参数；
6. 调用 `optimizer.zero_grad()` 清空上一批次的梯度。

In [ ]:
happy_history = train_binary(happy_model, happy_train_loader, happy_test_loader, epochs=10, lr=1e-3)


训练完成后，在评估阶段使用 `model.eval()` 和 `torch.no_grad()`，关闭 BatchNorm 的训练行为并避免保存梯度。

In [ ]:
happy_test_loss, happy_test_accuracy = evaluate_binary(happy_model, happy_test_loader, nn.BCELoss())
print(f"test loss = {happy_test_loss:.4f}, test accuracy = {happy_test_accuracy:.4f}")


虽然 `nn.Sequential` 很方便，但它只能表达线性的层连接。对于共享层、分支、跳跃连接或多输入多输出模型，需要使用自定义 `nn.Module`，在 `forward()` 中定义更灵活的数据流。

<a name='4'></a>
## 4 - 自定义 Module 与函数式 forward

这一部分使用更灵活的 PyTorch 模型写法，构建一个能够识别 6 个手语数字的卷积网络。PyTorch 中没有与 Keras Functional API 完全同名的接口，但自定义 `nn.Module` 的 `forward()` 可以清晰地表达同样的计算图思想。

- Sequential：层按照固定顺序连接；
- 自定义 Module：可以在 `forward()` 中使用分支、共享层和跳跃连接；
- PyTorch 的模型结构是在实际执行 `forward()` 时计算的。

<a name='4-1'></a>
### 4.1 - 加载 SIGNS 数据集

SIGNS 数据集包含代表数字 0 到 5 的 6 类手语图片。

In [ ]:
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_signs_dataset()


下面显示一张带标签的图片。可以修改 `index` 来查看其他样本。

In [ ]:
index = 9
plt.imshow(X_train_orig[index])
plt.title("y = " + str(np.squeeze(Y_train_orig[:, index])))
plt.axis("off")
plt.show()


<a name='4-2'></a>
### 4.2 - 划分训练集/测试集

由于输入是图像，使用 CNN 比全连接网络更自然。下面进行归一化，并将 NHWC 转为 PyTorch 的 NCHW。

注意：PyTorch 的 `CrossEntropyLoss` 需要类别索引标签，例如 `0, 1, ..., 5`，不需要 one-hot 标签。

In [ ]:
X_train = images_to_tensor(X_train_orig / 255.)
X_test = images_to_tensor(X_test_orig / 255.)
Y_train = class_labels_to_tensor(Y_train_orig)
Y_test = class_labels_to_tensor(Y_test_orig)

print("number of training examples =", X_train.shape[0])
print("number of test examples =", X_test.shape[0])
print("X_train shape =", tuple(X_train.shape))
print("Y_train shape =", tuple(Y_train.shape))
print("X_test shape =", tuple(X_test.shape))
print("Y_test shape =", tuple(Y_test.shape))


<a name='4-3'></a>
### 4.3 - 前向传播

下面的模型包含：

```text
CONV2D → RELU → MAXPOOL → CONV2D → RELU → MAXPOOL → FLATTEN → LINEAR
```

我们使用自定义 `nn.Module`，并在 `forward()` 中逐步计算。

- `nn.Conv2d` 完成卷积；
- `nn.ReLU` 完成逐元素 ReLU；
- `nn.MaxPool2d` 完成最大池化；
- `nn.Flatten` 将每个样本展平；
- `nn.Linear` 完成全连接映射。

原 Keras 版本最后使用 Softmax，而 PyTorch 的 `CrossEntropyLoss` 内部已经包含 LogSoftmax，因此模型最后应返回未归一化的 logits，不要在模型中额外添加 Softmax。

#### Window、kernel、filter、pool

在卷积中，kernel/filter 通常指卷积核。池化使用一个窗口在特征图上滑动，并对窗口内的值取最大值或平均值。

本模型中的 `same` 卷积使用显式的非对称 Zero Padding 来匹配 TensorFlow 的输出尺寸：偶数大小的卷积核无法只用一个对称 padding 完全复现 `same`，因此需要在上下或左右多填充一个像素。

<a name='ex-2'></a>
### 练习 2 - convolutional_model

实现 `convolutional_model`，构建以下模型：

```text
CONV2D → RELU → MAXPOOL → CONV2D → RELU → MAXPOOL → FLATTEN → LINEAR
```

参数设置：

- 第一层：8 个 `4 x 4` 滤波器，stride 为 1，`same` padding；
- 第一个池化层：`8 x 8` 窗口，stride 为 8；
- 第二层：16 个 `2 x 2` 滤波器，stride 为 1，`same` padding；
- 第二个池化层：`4 x 4` 窗口，stride 为 4；
- 展平后连接到 6 个输出神经元。

In [ ]:
# Exercise 2: convolutional_model

class SignConvNet(nn.Module):
    def __init__(self, input_shape):
        super().__init__()
        channels, height, width = input_shape
        self.pad1 = nn.ZeroPad2d((1, 2, 1, 2))  # TensorFlow SAME for 4x4, stride 1
        self.conv1 = nn.Conv2d(channels, 8, kernel_size=4, stride=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=8, stride=8)
        self.pad2 = nn.ZeroPad2d((0, 1, 0, 1))  # TensorFlow SAME for 2x2, stride 1
        self.conv2 = nn.Conv2d(8, 16, kernel_size=2, stride=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=4, stride=4)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(16 * 2 * 2, 6)

    def forward(self, x):
        x = self.pad1(x)
        x = self.relu1(self.conv1(x))
        x = self.pool1(x)
        x = self.pad2(x)
        x = self.relu2(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        return self.fc(x)  # logits，交给 CrossEntropyLoss 处理


def convolutional_model(input_shape):
    return SignConvNet(input_shape)


In [ ]:
conv_model = convolutional_model((3, 64, 64)).to(device)
model_summary(conv_model, (3, 64, 64))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(conv_model.parameters(), lr=1e-3)
print(criterion)
print(optimizer)


Sequential 和自定义 `nn.Module` 都会返回一个可以训练的 PyTorch 模型对象。区别在于：Sequential 只能表达线性层堆叠，而自定义 Module 可以在 `forward()` 中表达任意计算图。

<a name='4-4'></a>
### 4.4 - 训练模型

使用 `DataLoader` 以 minibatch 形式读取数据，并通过显式训练循环更新参数。

In [ ]:
sign_train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=64, shuffle=True)
sign_test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size=64, shuffle=False)
history = train_multiclass(conv_model, sign_train_loader, sign_test_loader, epochs=100, lr=1e-3)


<a name='5'></a>
## 5 - History 字典

PyTorch 不会自动返回 Keras 风格的 History 对象。本 notebook 在训练循环中手动把每个 epoch 的 loss 和 accuracy 保存到 `history` 字典中：

In [ ]:
history


下面可视化训练集和测试集上的 loss 与 accuracy。

In [ ]:
df_loss_acc = pd.DataFrame(history)
df_loss = df_loss_acc[["loss", "val_loss"]].rename(columns={"loss": "train", "val_loss": "validation"})
df_acc = df_loss_acc[["accuracy", "val_accuracy"]].rename(columns={"accuracy": "train", "val_accuracy": "validation"})
df_loss.plot(title="Model loss", figsize=(12, 8)).set(xlabel="Epoch", ylabel="Loss")
df_acc.plot(title="Model Accuracy", figsize=(12, 8)).set(xlabel="Epoch", ylabel="Accuracy")
plt.show()


**恭喜！** 你已经使用 PyTorch 构建了两个 CNN：一个识别笑脸，另一个识别手语数字。除此之外，你还理解了 `nn.Sequential` 与自定义 `nn.Module` 的使用场景，以及 PyTorch 训练循环的基本结构。

<a name='6'></a>
## 6 - 参考资料

建议阅读 PyTorch 官方文档：

- [torch.nn](https://pytorch.org/docs/stable/nn.html)
- [torch.nn.Conv2d](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)
- [torch.nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
- [torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html)
- [Training a classifier](https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)